# Comparação: Transformer vs RL (PPO) — Carteira Híbrida

Benchmark direto entre **TimeSeriesTransformer** (supervisionado) e **PPO Trading Agent** (RL)
no mesmo dataset híbrido (B3 + NASDAQ + Cripto), mesmas janelas temporais e métricas.

In [ ]:
import sys, os, time, warnings
from pathlib import Path
_root = Path(os.getcwd())
while not ((_root / 'src').exists() and (_root / 'pyproject.toml').exists()) and _root != _root.parent:
    _root = _root.parent
sys.path.insert(0, str(_root))
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader

from src.backtest.engine import BacktestEngine, BacktestConfig
from src.backtest.visualization import BG, PANEL, GOLD, GREEN, RED, WHITE, CYAN, PURPLE, ORANGE
from src.models.transformer_ts import TimeSeriesTransformer
from src.models.rl_env import TradingEnv, make_env
from stable_baselines3 import PPO
from stable_baselines3.common.vec_env import DummyVecEnv

pd.set_option('display.max_columns', 80)
pd.set_option('display.width', 160)
_images = _root / 'images'
_images.mkdir(exist_ok=True)

## 1. Dados Híbridos — B3 + NASDAQ + Cripto

In [ ]:
df = pd.read_parquet(_root / 'data/lse_market_data/combined_1d.parquet')
pivot = df.pivot_table(index='timestamp', columns='symbol', values='close')
valid_cols = pivot.columns[pivot.count() >= 300]
pivot = pivot[valid_cols]
pivot = pivot.ffill().bfill()
returns = pivot.pct_change().fillna(0)
std = returns.std() + 1e-8
returns_norm = (returns - returns.mean()) / std

split_idx = int(len(returns) * 0.7)
train_dates = returns.index[:split_idx]
test_dates = returns.index[split_idx:]

pd.DataFrame({
    '': ['tickers', 'total_days', 'train_days', 'test_days', 'train_from', 'test_from'],
    'value': [len(valid_cols), len(returns), split_idx, len(returns)-split_idx,
              str(train_dates[0].date()), str(test_dates[0].date())]
})

## 2. Transformer — Supervised Multivariate Forecast

In [ ]:
seq_len, pred_len = 20, 5
X = returns_norm.values
src_list, tgt_list, ret_tgt_list = [], [], []

for i in range(len(X) - seq_len - pred_len):
    src_list.append(X[i:i+seq_len])
    tgt_list.append(X[i+seq_len-1:i+seq_len+pred_len])
    ret_tgt_list.append(returns.values[i+seq_len])

src_tensor = torch.tensor(np.array(src_list), dtype=torch.float32)
tgt_tensor = torch.tensor(np.array(tgt_list), dtype=torch.float32)
actual_returns = np.array(ret_tgt_list)

src_train, tgt_train = src_tensor[:split_idx], tgt_tensor[:split_idx]
src_test, tgt_test = src_tensor[split_idx:], tgt_tensor[split_idx:]
ret_train = actual_returns[:split_idx]
ret_test = actual_returns[split_idx:]

train_ds = TensorDataset(src_train, tgt_train)
test_ds = TensorDataset(src_test, tgt_test)
train_loader = DataLoader(train_ds, batch_size=64, shuffle=True)
test_loader = DataLoader(test_ds, batch_size=64, shuffle=False)

input_dim = X.shape[1]
model_tf = TimeSeriesTransformer(input_dim=input_dim, d_model=64, n_heads=4,
                                 n_encoder_layers=2, n_decoder_layers=2,
                                 max_src_len=seq_len+10, max_tgt_len=pred_len+10)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model_tf = model_tf.to(device)

optimizer = optim.Adam(model_tf.parameters(), lr=1e-3)
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=10, gamma=0.5)
criterion = nn.MSELoss()

train_losses, test_losses = [], []
t0 = time.time()
for epoch in range(20):
    model_tf.train()
    total = 0
    for src, tgt in train_loader:
        src, tgt = src.to(device), tgt.to(device)
        optimizer.zero_grad()
        out = model_tf(src, tgt[:, :-1])
        loss = criterion(out, tgt[:, 1:])
        loss.backward()
        optimizer.step()
        total += loss.item()
    train_losses.append(total/len(train_loader))
    
    model_tf.eval()
    t_total = 0
    with torch.no_grad():
        for src, tgt in test_loader:
            src, tgt = src.to(device), tgt.to(device)
            out = model_tf(src, tgt[:, :-1])
            loss = criterion(out, tgt[:, 1:])
            t_total += loss.item()
    test_losses.append(t_total/len(test_loader))
    scheduler.step()

tf_train_time = time.time() - t0

def eval_tf(model, src_t, rets):
    model.eval()
    with torch.no_grad():
        preds = model.predict(src_t.to(device), steps=1)[:, 0, :].cpu().numpy()
    exp_p = np.exp(preds * 10)
    w = exp_p / exp_p.sum(axis=1, keepdims=True)
    port_ret = (w * rets).sum(axis=1)
    cum = np.cumprod(1 + port_ret)
    sharpe = np.mean(port_ret) / (np.std(port_ret) + 1e-8) * np.sqrt(252)
    dd = (cum / np.maximum.accumulate(cum) - 1).min() * 100
    return {'return_pct': (cum[-1]-1)*100, 'sharpe': sharpe, 'max_dd_pct': dd,
            'port_ret': port_ret, 'cum': cum}

tf_train = eval_tf(model_tf, src_train, ret_train)
tf_test = eval_tf(model_tf, src_test, ret_test)

pd.DataFrame({
    'metric': ['Train MSE', 'Test MSE', 'Train Return', 'Test Return', 'Train Sharpe', 'Test Sharpe', 'Train DD', 'Test DD'],
    'value': [f'{train_losses[-1]:.4f}', f'{test_losses[-1]:.4f}',
              f"{tf_train['return_pct']:+.2f}%", f"{tf_test['return_pct']:+.2f}%",
              f"{tf_train['sharpe']:.3f}", f"{tf_test['sharpe']:.3f}",
              f"{tf_train['max_dd_pct']:.1f}%", f"{tf_test['max_dd_pct']:.1f}%"]
})

## 3. RL (PPO) — Trading Agent

In [ ]:
def make_rl_df(returns_slice):
    port_ret = returns_slice.mean(axis=1)
    price = (1 + port_ret).cumprod() * 100
    return pd.DataFrame({'close': price})

df_rl_train = make_rl_df(returns.values[:split_idx])
df_rl_test = make_rl_df(returns.values[split_idx:])

t0 = time.time()
env_train = make_env(df_rl_train, preset='balanced', lookback=20)
vec_env = DummyVecEnv([lambda: env_train])
model_ppo = PPO('MlpPolicy', vec_env, learning_rate=3e-4, verbose=0, device='auto',
                n_steps=2048, batch_size=256, gamma=0.99, ent_coef=0.01)
model_ppo.learn(total_timesteps=50_000, progress_bar=False)
rl_train_time = time.time() - t0

def eval_ppo(model, df_eval):
    env = make_env(df_eval, preset='balanced', lookback=20)
    obs, _ = env.reset()
    done = False
    pv = [env.initial_balance]
    while not done:
        action, _ = model.predict(obs, deterministic=True)
        obs, reward, done, trunc, info = env.step(int(action))
        pv.append(info['portfolio_value'])
        done = done or trunc
    pv = np.array(pv)
    daily = np.diff(pv) / pv[:-1]
    cum = np.cumprod(1 + daily)
    sharpe = np.mean(daily) / (np.std(daily) + 1e-8) * np.sqrt(252)
    dd = (cum / np.maximum.accumulate(cum) - 1).min() * 100
    return {'return_pct': (cum[-1]-1)*100, 'sharpe': sharpe, 'max_dd_pct': dd,
            'port_ret': daily, 'cum': cum}

rl_train = eval_ppo(model_ppo, df_rl_train)
rl_test = eval_ppo(model_ppo, df_rl_test)

pd.DataFrame({
    'metric': ['Train Return', 'Test Return', 'Train Sharpe', 'Test Sharpe', 'Train DD', 'Test DD'],
    'value': [f"{rl_train['return_pct']:+.2f}%", f"{rl_test['return_pct']:+.2f}%",
              f"{rl_train['sharpe']:.3f}", f"{rl_test['sharpe']:.3f}",
              f"{rl_train['max_dd_pct']:.1f}%", f"{rl_test['max_dd_pct']:.1f}%"]
})

## 4. Buy & Hold Baseline

In [ ]:
bh_train_ret = returns.values[:split_idx].mean(axis=1)
bh_test_ret = returns.values[split_idx:].mean(axis=1)
bh_train_cum = np.cumprod(1 + bh_train_ret)
bh_test_cum = np.cumprod(1 + bh_test_ret)
bh_train_sharpe = np.mean(bh_train_ret) / (np.std(bh_train_ret) + 1e-8) * np.sqrt(252)
bh_test_sharpe = np.mean(bh_test_ret) / (np.std(bh_test_ret) + 1e-8) * np.sqrt(252)
bh_train_dd = (bh_train_cum / np.maximum.accumulate(bh_train_cum) - 1).min() * 100
bh_test_dd = (bh_test_cum / np.maximum.accumulate(bh_test_cum) - 1).min() * 100

pd.DataFrame({
    'metric': ['Train Return', 'Test Return', 'Train Sharpe', 'Test Sharpe', 'Train DD', 'Test DD'],
    'value': [f"{(bh_train_cum[-1]-1)*100:+.2f}%", f"{(bh_test_cum[-1]-1)*100:+.2f}%",
              f"{bh_train_sharpe:.3f}", f"{bh_test_sharpe:.3f}",
              f"{bh_train_dd:.1f}%", f"{bh_test_dd:.1f}%"]
})

## 5. Comparação Final — Transformer vs RL vs BH

In [ ]:
comp = pd.DataFrame([
    {'model': 'Transformer', 'split': 'train', 'return_pct': tf_train['return_pct'],
     'sharpe': tf_train['sharpe'], 'max_dd_pct': tf_train['max_dd_pct'],
     'bh_return_pct': (bh_train_cum[-1]-1)*100, 'excess_pct': tf_train['return_pct'] - (bh_train_cum[-1]-1)*100},
    {'model': 'Transformer', 'split': 'test', 'return_pct': tf_test['return_pct'],
     'sharpe': tf_test['sharpe'], 'max_dd_pct': tf_test['max_dd_pct'],
     'bh_return_pct': (bh_test_cum[-1]-1)*100, 'excess_pct': tf_test['return_pct'] - (bh_test_cum[-1]-1)*100},
    {'model': 'PPO (RL)', 'split': 'train', 'return_pct': rl_train['return_pct'],
     'sharpe': rl_train['sharpe'], 'max_dd_pct': rl_train['max_dd_pct'],
     'bh_return_pct': (bh_train_cum[-1]-1)*100, 'excess_pct': rl_train['return_pct'] - (bh_train_cum[-1]-1)*100},
    {'model': 'PPO (RL)', 'split': 'test', 'return_pct': rl_test['return_pct'],
     'sharpe': rl_test['sharpe'], 'max_dd_pct': rl_test['max_dd_pct'],
     'bh_return_pct': (bh_test_cum[-1]-1)*100, 'excess_pct': rl_test['return_pct'] - (bh_test_cum[-1]-1)*100},
    {'model': 'Buy & Hold', 'split': 'train', 'return_pct': (bh_train_cum[-1]-1)*100,
     'sharpe': bh_train_sharpe, 'max_dd_pct': bh_train_dd, 'bh_return_pct': 0, 'excess_pct': 0},
    {'model': 'Buy & Hold', 'split': 'test', 'return_pct': (bh_test_cum[-1]-1)*100,
     'sharpe': bh_test_sharpe, 'max_dd_pct': bh_test_dd, 'bh_return_pct': 0, 'excess_pct': 0},
])

comp['train_time_s'] = comp['model'].map({
    'Transformer': round(tf_train_time, 1),
    'PPO (RL)': round(rl_train_time, 1),
    'Buy & Hold': 0
})

comp_test = comp[comp['split'] == 'test'].sort_values('excess_pct', ascending=False).reset_index(drop=True)
comp_test[['model', 'return_pct', 'sharpe', 'max_dd_pct', 'excess_pct', 'train_time_s']]

## 6. Equity Curves Comparativas

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(24, 9))
fig.patch.set_facecolor(BG)

ax = axes[0]
ax.set_facecolor(PANEL)
ax.plot(bh_train_cum, color=WHITE, lw=1.5, ls='--', alpha=0.5, label='Buy & Hold')
ax.plot(tf_train['cum'], color=GOLD, lw=2, label='Transformer')
ax.plot(rl_train['cum'], color=CYAN, lw=2, label='PPO (RL)')
ax.axhline(1, color='white', ls='--', alpha=0.15)
ax.set_title('Training Set — Equity Curves (Normalized)', color='white', fontsize=14, fontweight='bold')
ax.tick_params(colors='white', labelsize=12)
ax.grid(True, alpha=0.12)
ax.legend(fontsize=12, facecolor=PANEL, edgecolor='white', labelcolor='white')

ax = axes[1]
ax.set_facecolor(PANEL)
ax.plot(bh_test_cum, color=WHITE, lw=1.5, ls='--', alpha=0.5, label='Buy & Hold')
ax.plot(tf_test['cum'], color=GOLD, lw=2, label='Transformer')
ax.plot(rl_test['cum'], color=CYAN, lw=2, label='PPO (RL)')
ax.axhline(1, color='white', ls='--', alpha=0.15)
ax.set_title('Test Set — Equity Curves (Normalized)', color='white', fontsize=14, fontweight='bold')
ax.tick_params(colors='white', labelsize=12)
ax.grid(True, alpha=0.12)
ax.legend(fontsize=12, facecolor=PANEL, edgecolor='white', labelcolor='white')

fig.suptitle('Transformer vs RL vs Buy & Hold — Hybrid Portfolio (27 assets)', color='white', fontsize=16, fontweight='bold', y=0.98)
fig.tight_layout(rect=[0, 0, 1, 0.95])
fig.savefig(_images / 'tf_vs_rl_equity.png', dpi=300, facecolor=BG, edgecolor='none', bbox_inches='tight')
plt.show()

## 7. Overfitting Diagnostics — Bar Chart

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(24, 7))
fig.patch.set_facecolor(BG)

models = ['Transformer', 'PPO (RL)', 'Buy & Hold']
colors = [GOLD, CYAN, WHITE]

ax = axes[0]
ax.set_facecolor(PANEL)
test_rets = [tf_test['return_pct'], rl_test['return_pct'], (bh_test_cum[-1]-1)*100]
bars = ax.bar(models, test_rets, color=[GREEN if v > 0 else RED for v in test_rets], alpha=0.85)
ax.axhline(0, color='white', lw=0.5)
ax.set_title('Test Return (%)', color='white', fontsize=14, fontweight='bold')
ax.tick_params(colors='white', labelsize=12)
ax.grid(True, alpha=0.12, axis='y')

ax = axes[1]
ax.set_facecolor(PANEL)
test_sharpes = [tf_test['sharpe'], rl_test['sharpe'], bh_test_sharpe]
ax.bar(models, test_sharpes, color=colors, alpha=0.85)
ax.axhline(0, color='white', lw=0.5)
ax.set_title('Test Sharpe', color='white', fontsize=14, fontweight='bold')
ax.tick_params(colors='white', labelsize=12)
ax.grid(True, alpha=0.12, axis='y')

ax = axes[2]
ax.set_facecolor(PANEL)
gaps = [tf_train['return_pct'] - tf_test['return_pct'],
        rl_train['return_pct'] - rl_test['return_pct'], 0]
ax.bar(models, gaps, color=[GOLD, CYAN, WHITE], alpha=0.85)
ax.axhline(0, color='white', lw=0.5)
ax.set_title('Train-Test Gap (Overfitting Indicator)', color='white', fontsize=14, fontweight='bold')
ax.tick_params(colors='white', labelsize=12)
ax.grid(True, alpha=0.12, axis='y')

fig.suptitle('Overfitting Diagnostics — Model Comparison', color='white', fontsize=16, fontweight='bold', y=0.98)
fig.tight_layout(rect=[0, 0, 1, 0.95])
fig.savefig(_images / 'tf_vs_rl_overfitting.png', dpi=300, facecolor=BG, edgecolor='none', bbox_inches='tight')
plt.show()

## 8. Resumo Executivo

In [ ]:
summary = pd.DataFrame([
    {'model': 'Transformer', 'category': 'Supervised DL',
     'test_return': tf_test['return_pct'], 'test_sharpe': tf_test['sharpe'],
     'overfitting': 'EXTREMO' if (tf_train['return_pct'] - tf_test['return_pct']) > 100 else 'LEVE',
     'train_time_s': round(tf_train_time, 1), 'interpretability': 'Baixa',
     'recommendation': 'Regularizar (dropout, early stop)'},
    {'model': 'PPO (RL)', 'category': 'Reinforcement Learning',
     'test_return': rl_test['return_pct'], 'test_sharpe': rl_test['sharpe'],
     'overfitting': 'LEVE' if (rl_train['return_pct'] - rl_test['return_pct']) < 50 else 'MODERADO',
     'train_time_s': round(rl_train_time, 1), 'interpretability': 'Média (ações discretas)',
     'recommendation': 'Mais timesteps + ensemble seeds'},
    {'model': 'Buy & Hold', 'category': 'Baseline',
     'test_return': (bh_test_cum[-1]-1)*100, 'test_sharpe': bh_test_sharpe,
     'overfitting': 'N/A', 'train_time_s': 0, 'interpretability': 'Total',
     'recommendation': 'Baseline mínimo a superar'},
])
summary